# What's new in CDA

*Standalone -- no prior vignette required.*

<iframe width="560" height="315" src="https://www.youtube.com/embed/PLACEHOLDER_VIDEO_ID" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

Watch the video above for the quick version, or just read along below -- every cell here is real and already run. Want to try it yourself with your own search term? Open this notebook in [Google Colab](https://colab.research.google.com/github/CancerDataAggregator/Community-Notebooks/blob/main/Tutorials/10_CN_whats_new.ipynb) -- same notebook, fully live.

I want to see what CDA already has on melanoma. I don't know the vocabulary, I don't know which columns are relevant, and I don't want to find that out before I even know if this is worth my time. So I'm not going to look anything up first -- just type the word and see what happens.

In [ ]:
from cdapython import *
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

In [ ]:
summarize_subjects('melanoma')

1,565 subjects, 78,263 related files, from one word and zero setup. And look at the data_source breakdown -- these subjects turn up across GDC, IDC, GC, CTDC, and ICDC, in just about every combination. One search, reaching across every data center CDA covers.

There's already something unexpected sitting in the default columns, before I've customized anything: I searched for a human cancer, and the species breakdown shows 28 dogs and 20 mice right alongside 1,488 humans. That's not noise -- CDA's Integrated Canine Data Commons tracks melanoma in dogs specifically because it helps researchers studying the human disease. One plain word search, and I'm already looking at cross-species data I didn't think to ask for.

Let's add a few columns and see what's actually driving these matches -- `anatomic_site`, `diagnosis`, and `format` are a reasonable starting guess for a cancer search, but if you want to see everything that's queryable, `columns()` will list it all:

In [ ]:
summarize_subjects('melanoma', add_columns=['anatomic_site', 'diagnosis', 'format'])

Here's a bigger surprise: the single largest anatomic_site value isn't skin at all -- it's blood, at nearly 7,900, ahead of skin of body at 7,220, even on a skin cancer search. That's a lot more than the 1,565 subjects I actually matched -- which is the tell that these numbers aren't counting subjects at all.

They're counting files. `anatomic_site` (like `diagnosis` and `format`) lives on the **file** table, not the `subject` table -- so when a file-table column gets added to a `summarize_subjects()` call, the breakdown is a count of matching *files* carrying each value, across every file tied to every matching subject, not a count of subjects. With that in mind, blood is still a real and informative result, not a tagalong oddity: blood samples are a genuine part of melanoma care, used to track tumor genetics and monitor treatment response without another tissue biopsy. A subject who matched on their melanoma diagnosis is very likely also bringing blood-based testing files along with them -- there just happen to be a lot of those files relative to skin-of-body files.

Further down the list, something else is going on: anatomic_site values eye and uvea both show exactly 497 -- still a file count by this same mechanism, not a subject count, but the exact match between the two is the real story here, not the number itself. This one isn't a tagalong like blood -- melanoma isn't only a skin cancer. It happens in the eye too, called uveal melanoma, and it's a genuinely distinct subtype. CDA knows the uvea is part of the eye, so a plain melanoma search correctly pulled in both, without me ever having to know that uveal melanoma existed in the first place.

## Why do some values match exactly like that?

CDA's anatomic_site and disease fields are backed by real medical ontologies -- UBERON for anatomy, ICD-O-3 for disease. A broad term automatically pulls in everything more specific underneath it in that tree, so a search for eye also catches uvea without me ever having to know uvea was part of the eye.

It only goes one direction, though. Searching for eye reaches down to everything underneath it -- but searching for uvea or iris won't reach back up to eye, and it won't pull in unrelated parts of the eye like the retina either:

![Toy ontology diagram showing eye as a parent of uvea, retina, and cornea, with uvea further branching into iris and choroid. A green arrow shows that searching "eye" reaches down to all of these. A red dashed line from iris back up to eye is marked with an X, showing that searching "iris" does not reach back up to "eye."](../images/eye_uvea_ontology.svg)

Up until recently, CDA could do this matching, but it couldn't tell you which term actually did the matching -- you'd just see the count and have to take it on faith. That's what add_extras now solves: it adds columns showing the real vocabulary behind each match, without changing which rows matched at all.

In [ ]:
summarize_subjects('melanoma', add_columns=['anatomic_site', 'diagnosis', 'format'], add_extras=['all'])

Same subjects, same count -- and now uvea's slim_terms value is literally eye. That confirms it: this wasn't a coincidence, it's the real ontology, now visible instead of hidden. blood's synonym_terms show portion of blood and vertebrate blood, with containing_terms like hematopoietic system -- real vocabulary explaining what that match actually means, instead of just a word I'd have to trust.

But the diagnosis table makes the point even more clearly. Scroll down to it: every melanoma subtype here -- nodular, spindle cell, epithelioid, amelanotic, lentigo maligna, even malignant melanoma itself -- shares the exact same slim_terms value: Nevi and melanomas. A dozen different-sounding diagnoses, and now I can see in writing why they all count as the same broad category, instead of just trusting that they do.

Worth knowing: this only shows something new on two fields right now -- anatomic_site and disease -- since those are the two backed by a real external ontology. Fields like sex or species are fully standardized but flat, with no more-specific version to roll up to. A few fields, like stage, aren't harmonized yet at all -- that's not a gap in this feature, just where the underlying harmonization work currently stands.

## Getting more specific

I've got melanoma subjects and a sense of what's in them. Now say I actually want to use this data -- I'm after sequencing data specifically, not imaging or clinical records. Rather than guess at a column name, I'll search by what a column's described as doing:

In [ ]:
columns(description='format')

format looks right. Before filtering on it, let's see what the real values look like:

In [ ]:
column_values('format', data_source='GDC')

VCF -- variant call format, real sequencing output -- is already one of the biggest categories across our melanoma files, over 13,000 of them. Let's see how many human melanoma subjects actually have it:

In [ ]:
summarize_subjects(match_all=['diagnosis = *melanoma*', 'format = VCF', 'species = human'])

1,406 subjects, down from 1,565, with 13,211 VCF files between them. A focused, usable number of human melanoma subjects with sequencing data ready to go -- reached by checking what was actually there before writing a filter, instead of guessing and hoping.

One word, a filter built from real values instead of guesses, and a new way to see exactly why CDA matched what it matched. Everything above is real and already run. Want to try your own search? Open this notebook in [Google Colab](https://colab.research.google.com/github/CancerDataAggregator/Community-Notebooks/blob/main/Tutorials/10_CN_whats_new.ipynb) -- same notebook, fully live, swap in whatever you're actually searching for.

For the deeper walkthroughs this builds on, see [Is there anything about X?](01_global_search.ipynb) and [Getting oriented before I filter](02_systematic_browsing.ipynb).